In [0]:
# Databricks notebook source
# Validates the landing zone before the medallion pipelines run.
# Fails loudly if a source directory is empty or missing.

from pyspark.sql.functions import col

LANDING = "/Volumes/ecom_dev/landing/raw_files"

SOURCES = [
    "orders", "order_items", "customers", "sellers", "products",
    "payments", "reviews", "geolocation", "category_translation",
]

def count_csvs(path):
    """Recursive count — Auto Loader reads nested folders, so this must too."""
    total = 0
    for entry in dbutils.fs.ls(path):
        if entry.isDir():
            total += count_csvs(entry.path)
        elif entry.name.endswith(".csv"):
            total += 1
    return total

results, failures = [], []

for src in SOURCES:
    try:
        n = count_csvs(f"{LANDING}/{src}/")
        results.append((src, n))
        if n == 0:
            failures.append(f"{src}: no CSV files found")
    except Exception as e:
        results.append((src, -1))
        failures.append(f"{src}: {type(e).__name__}")

print(f"{'source':24} {'files':>6}")
print("-" * 32)
for src, n in results:
    flag = "  <-- EMPTY" if n == 0 else ""
    print(f"{src:24} {n:>6}{flag}")

if failures:
    raise Exception("Landing validation failed:\n  " + "\n  ".join(failures))

print(f"\nOK — {sum(n for _, n in results)} files across {len(SOURCES)} sources")
dbutils.jobs.taskValues.set(key="total_files", value=sum(n for _, n in results))